In [ ]:
# @title
import torch
import torchtune.modules as Module
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.amp as amp
import tiktoken
import datasets
from datasets import load_dataset

ds = load_dataset("ajibawa-2023/Children-Stories-Collection")


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision('high')
scaler = amp.GradScaler()

In [4]:
# # @title
d_set = ''.join(ds['train']['text'])
len(d_set)

1829045733

In [5]:
# @title
class Helper():
  def __init__(self, train_ds):
    self.encoder = tiktoken.get_encoding("r50k_base")
    self.vocab_size = self.encoder.n_vocab # fixed vocab size being train_ds not chrs
    self.enc_ds = self.encode(train_ds)

  def encode(self, str):
    return torch.tensor(self.encoder.encode(str))

  # randomly samples (B, T) -sized data from training dataset
  def dataloader(self, B, T):
    ix = torch.randint(len(self.enc_ds) - T, (B,))
    indices = ix.view(-1, 1) + torch.arange(T)
    X = self.enc_ds[indices]
    y = self.enc_ds[indices+1]
    return X.to(device), y.to(device)

  def training(self, model, iterations, batch_size, seq_len):
    for epoch in range(iterations):
      X, y = self.dataloader(batch_size, seq_len)
      with amp.autocast(device_type="cuda"):
        output = model.forward(X, targets=y)


In [ ]:
# @title
class Attention_Head(nn.Module):
  def __init__(self, embed_dims, head_size, num_heads=16):
    super().__init__()
    self.embed_dims = embed_dims
    self.num_heads = num_heads
    self.head_size = head_size
    self.total_heads = head_size * num_heads

    self.q_proj = nn.Linear(embed_dims, self.total_heads)
    self.k_proj = nn.Linear(embed_dims, self.total_heads)
    self.v_proj = nn.Linear(embed_dims, self.total_heads)

    self.pe = Module.RotaryPositionalEmbeddings(self.head_size)

  def forward(self, logits, batch_size, seq_len):
    q = self.q_proj(logits).view(batch_size, seq_len, self.num_heads, self.head_size).transpose(1,2)
    k = self.k_proj(logits).view(batch_size, seq_len, self.num_heads, self.head_size).transpose(1,2)

    q_pe = self.pe(q)
    k_pe = self.pe(k)

    v = self.v_proj(logits).view(batch_size, seq_len, self.num_heads, self.head_size).transpose(1,2)

    attention_out = F.scaled_dot_product_attention(q_pe, k_pe, v, is_causal=True)
    out = attention_out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.total_heads)
    return out

class FFN(nn.Sequential):
  def __init__(self, embed_dims, hidden_dim):
    super().__init__(
        nn.Linear(embed_dims, hidden_dim),
        nn.SiLU(),
        nn.Linear(hidden_dim, embed_dims)
    )

class Block(nn.Module):
  def __init__(self, embed_dims, head_size):
    super().__init__()
    self.embed_dims = embed_dims
    self.head_size = head_size

    self.rms_Norm1 = nn.RMSNorm(embed_dims)
    self.rms_Norm2 = nn.RMSNorm(embed_dims)

    self.Attention_Head = Attention_Head(embed_dims, head_size)
    self.FFN = FFN(embed_dims, embed_dims*4)

  def forward(self, logits, batch_size, seq_len):

    x = self.Attention_Head(self.rms_Norm1(logits), batch_size, seq_len)
    # added residual connections
    x = x + logits
    out = self.FFN(self.rms_Norm2(x))
    # added residual connections
    out = out + x
    return out

class Model(nn.Module):
  def __init__(self, batch_size, seq_len, embed_dims, head_size, block_num, vocab_size, lr):
    super().__init__()
    self.batch_size = batch_size
    self.seq_len = seq_len
    self.embed_dims = embed_dims
    self.head_size = head_size
    self.vocab_size = vocab_size

    self.token_embed = nn.Embedding(vocab_size, embed_dims)
    self.rms_Norm_embed = nn.RMSNorm(embed_dims)
    self.embed_proj = nn.Linear(embed_dims, vocab_size)
    self.block_list = nn.ModuleList([Block(embed_dims, head_size) for _ in range(block_num)])

    self.optimizer = optim.AdamW(self.parameters(), lr=lr)
    self.scheduler = optim.lr_scheduler.OneCycleLR(self.optimizer, 1e-3, total_steps=10000, pct_start=0.1, anneal_strategy='cos', final_div_factor=100.0)

  def forward(self, input, targets=None):
    batch_size, seq_len = input.shape
    logits = self.token_embed(input)


    for block in self.block_list:
      logits = block(logits, batch_size, seq_len)

    unembed_out = self.embed_proj(self.rms_Norm_embed(logits))

    if targets != None:
      preds = unembed_out.view(self.batch_size*self.seq_len, -1)
      targets = targets.view(-1)

      loss_fn = F.cross_entropy(preds, targets)
      print(loss_fn.item())
      self.optimizer.zero_grad()
      scaler.scale(loss_fn).backward()
      scaler.unscale_(self.optimizer)
      nn.utils.clip_grad_norm_(self.parameters(), 1.0)
      scaler.step(self.optimizer)
      self.scheduler.step()
      scaler.update()
      print(loss_fn.item())

    return unembed_out

  def generate(self, input_tokens, max_tokens):
      for _ in range(max_tokens):
          last_seq = input_tokens[:, -self.seq_len:]
          logits = self(last_seq)
          logits = logits[:, -1, :]
          probs = F.softmax(logits, dim=-1)
          next_tok = torch.multinomial(probs, num_samples=1)
          input_tokens = torch.cat((input_tokens, next_tok), dim=1)
      return input_tokens

In [7]:
helper = Helper(d_set)

In [30]:
batch_size = 16
seq_len = 1024
embed_dims = 1024
head_size = 64
block_num = 12
vocab_size = helper.vocab_size
lr = 1e-3

model = Model(batch_size, seq_len, embed_dims, head_size, block_num, vocab_size, lr)
model = model.to(device)

# model.load_state_dict(torch.load("workspace/server/children_stories_model_weights.pt"))

In [ ]:
helper.training(model, 10000, batch_size, seq_len)

In [ ]:
input = helper.encode("lorem ipsum ")
input_toks = torch.unsqueeze(input, dim=0).to(device)
out_toks = model.generate(input_toks, 50)
print(helper.encoder.decode(out_toks[0].tolist()))

In [ ]:
torch.save(model.state_dict(), "workspace/server/children_stories_model_weights2.pt")